# Đo "data collection considerations" bằng số: so sánh bốn nguồn thật

Phần 5 của lý thuyết liệt kê câu hỏi phải trả lời trước khi viết pipeline: **volume**, **velocity**,
**chất lượng** (thiếu, trùng, sentinel), **độ đều** (có lỗ hổng thời gian không). Thay vì đoán, bài này
viết một bộ **profiler** dùng lại được và chạy nó trên các nguồn thật của module:

| Nguồn | Loại | Có từ đâu |
| --- | --- | --- |
| `nasa-http-logs` | log | `ensure()` tự tải |
| `air-quality-uci` | IoT | `ensure()` tự tải |
| `open-meteo` | API | bài basic 01 (nếu đã chạy) |
| `usgs-earthquakes` | API | bài intermediate 01 (nếu đã chạy) |

Các hàm nạp **đã viết sẵn** (bản tối giản, chỉ để có bảng mà đo). Bạn viết profiler và phần đo dung lượng,
rồi dùng kết quả viết báo cáo khuyến nghị cách thu thập cho từng nguồn. Chấm:
`uv run pytest tracks/data-engineer/modules/00-introduction/exercises/intermediate/02-source-profiling -q`

In [ ]:
# explore: tìm gốc repo, tải hai bộ tĩnh nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

RAW = {"nasa-http-logs": ensure("nasa-http-logs"), "air-quality-uci": ensure("air-quality-uci"),
       "open-meteo": ROOT / "data" / "raw" / "open-meteo",
       "usgs-earthquakes": ROOT / "data" / "raw" / "usgs-earthquakes"}
CLEAN = ROOT / "data" / "clean" / "source-profiles"
RAW

In [ ]:
import gzip
import json
import zipfile
from pathlib import Path

import pandas as pd

FMT = "%Y-%m-%d %H:%M:%S"


# ---- Hàm nạp viết sẵn (tối giản, không phải bài này) ----

def load_nasa(path: Path) -> pd.DataFrame:
    """Log NASA: ts (UTC), host, request, status, bytes. Dòng không khớp regex bị bỏ."""
    with gzip.open(path, "rt", encoding="latin-1") as fh:
        lines = pd.Series(fh.read().split("\n"))
    df = lines.str.extract(r'^(?P<host>\S+) \S+ \S+ \[(?P<ts>[^\]]+)\] "(?P<request>.*)" '
                           r'(?P<status>\d{3}) (?P<bytes>\d+|-)\s*$').dropna(subset=["host"])
    # cả tháng 7 offset đều là -0400: cộng 4 giờ nhanh hơn nhiều so với đọc %z
    df["ts"] = (pd.to_datetime(df["ts"].str.slice(0, 20), format="%d/%b/%Y:%H:%M:%S")
                + pd.Timedelta(hours=4)).dt.tz_localize("UTC")
    df["status"] = df["status"].astype(int)
    df["bytes"] = pd.to_numeric(df["bytes"].replace("-", "0"))
    return df[["ts", "host", "request", "status", "bytes"]].reset_index(drop=True)


def load_air(zip_path: Path) -> pd.DataFrame:
    """UCI Air Quality: ts + 13 cột số đo, GIỮ NGUYÊN -200 (để profiler tự đếm sentinel)."""
    with zipfile.ZipFile(zip_path) as zf, zf.open("AirQualityUCI.csv") as fh:
        raw = pd.read_csv(fh, sep=";", decimal=",").dropna(how="all").dropna(axis=1, how="all")
    ts = pd.to_datetime(raw["Date"] + " " + raw["Time"], format="%d/%m/%Y %H.%M.%S")
    return pd.concat([ts.rename("ts"), raw.drop(columns=["Date", "Time"])], axis=1).reset_index(drop=True)


def load_weather(raw_dir: Path) -> pd.DataFrame:
    """Các tệp JSON Open-Meteo của bài basic 01."""
    frames = []
    for path in sorted(Path(raw_dir).glob("*.json")):
        daily = pd.DataFrame(json.loads(path.read_text(encoding="utf-8"))["daily"])
        frames.append(daily.assign(city=path.name.split("_")[0]))
    df = pd.concat(frames, ignore_index=True)
    df["ts"] = pd.to_datetime(df.pop("time"))
    return df


def load_usgs(raw_dir: Path) -> pd.DataFrame:
    """Mọi trang của bài intermediate 01, CHƯA dedup (để profiler thấy bản ghi lấy lại)."""
    rows = []
    for page in sorted(Path(raw_dir).glob("run-*/page-*.json")):
        for f in json.loads(page.read_text(encoding="utf-8"))["features"]:
            rows.append({"id": f["id"], "ts": f["properties"]["time"], "updated": f["properties"]["updated"],
                         "mag": f["properties"]["mag"], "place": f["properties"]["place"]})
    df = pd.DataFrame(rows)
    df["ts"] = pd.to_datetime(df["ts"], unit="ms", utc=True)
    df["updated"] = pd.to_datetime(df["updated"], unit="ms", utc=True)
    return df

## 1. Profiler

`profile(df, ts_col, key=None, sentinels=()) -> dict` không sửa `df`, không giả định `df` đã sắp xếp:

| Khoá | Nghĩa |
| --- | --- |
| `rows`, `columns` | số dòng, số cột (kể cả `ts_col`) |
| `start`, `end` | `ts` nhỏ nhất, lớn nhất, dạng `"YYYY-MM-DD HH:MM:SS"` |
| `span_days`, `rows_per_day` | (end − start) tính bằng ngày (số thực); rows / span_days (**velocity**) |
| `median_interval_s`, `max_gap_s`, `max_gap_start` | trung vị và lớn nhất của khoảng cách giữa hai `ts` liên tiếp (giây, sau khi sắp), thời điểm bắt đầu khoảng lớn nhất |
| `duplicate_rows` | số dòng giống hệt một dòng trước nó (mọi cột) |
| `duplicate_keys` | số dòng trùng `key` (danh sách cột) với một dòng trước; `None` nếu không truyền `key` |
| `sentinel_count` | số ô (ngoài `ts_col`) có giá trị nằm trong `sentinels` |
| `worst_null_column`, `worst_null_share` | cột (ngoài `ts_col`) có tỉ lệ thiếu cao nhất và tỉ lệ đó; ô là sentinel tính là thiếu |

`gaps(df, ts_col, min_gap_s) -> DataFrame` cột `start`, `end`, `seconds`: mọi khoảng giữa hai `ts` liên tiếp
dài **hơn** `min_gap_s` giây, sắp theo `seconds` giảm dần.

In [ ]:
def profile(df: pd.DataFrame, ts_col: str, key: list[str] | None = None, sentinels=()) -> dict:
    """Các khoá trong bảng trên."""
    # TODO
    raise NotImplementedError


def gaps(df: pd.DataFrame, ts_col: str, min_gap_s: float) -> pd.DataFrame:
    """Các khoảng lặng > min_gap_s giây: cột start, end, seconds; dài nhất trước."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: nạp và profile các nguồn đang có
frames = {
    "nasa-http-logs": (load_nasa(RAW["nasa-http-logs"] / "NASA_access_log_Jul95.gz"), {}),
    "air-quality-uci": (load_air(RAW["air-quality-uci"] / "air-quality.zip"), {"key": ["ts"], "sentinels": (-200,)}),
}
if list(RAW["open-meteo"].glob("*.json")):
    frames["open-meteo"] = (load_weather(RAW["open-meteo"]), {"key": ["city", "ts"]})
if list(RAW["usgs-earthquakes"].glob("run-*/page-*.json")):
    frames["usgs-earthquakes"] = (load_usgs(RAW["usgs-earthquakes"]), {"key": ["id"]})

PROFILES = {name: profile(df, "ts", **kw) for name, (df, kw) in frames.items()}
pd.DataFrame(PROFILES)

In [ ]:
# explore: lỗ hổng thời gian > 10 phút trong log, > 1 giờ trong IoT
print(gaps(frames["nasa-http-logs"][0], "ts", 600))
print(gaps(frames["air-quality-uci"][0], "ts", 3600))

## 2. Dung lượng: thô vs Parquet, và ngoại suy một năm

- `footprint(df, raw_bytes, path) -> dict`: ghi đè `df` ra Parquet tại `path` (nén `zstd`), trả
  `raw_bytes`, `parquet_bytes` (cỡ tệp vừa ghi), `raw_bytes_per_row`, `parquet_bytes_per_row`,
  `compression_ratio` (= raw / parquet).
- `yearly_volume_gb(rows_per_day, bytes_per_row) -> float`: ngoại suy dung lượng 365 ngày, GB = 1e9 byte.

In [ ]:
def footprint(df: pd.DataFrame, raw_bytes: int, path: Path) -> dict:
    """Ghi Parquet zstd (ghi đè), trả raw_bytes, parquet_bytes, *_per_row, compression_ratio."""
    # TODO
    raise NotImplementedError


def yearly_volume_gb(rows_per_day: float, bytes_per_row: float) -> float:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: cỡ thô (không nén) của từng nguồn, đo dung lượng
def raw_size(name: str) -> int:
    if name == "nasa-http-logs":
        with gzip.open(RAW[name] / "NASA_access_log_Jul95.gz") as fh:
            return sum(len(chunk) for chunk in iter(lambda: fh.read(1 << 20), b""))
    if name == "air-quality-uci":
        return zipfile.ZipFile(RAW[name] / "air-quality.zip").getinfo("AirQualityUCI.csv").file_size
    return sum(p.stat().st_size for p in RAW[name].rglob("*.json") if p.name != "state.json")

for name, (df, _) in frames.items():
    PROFILES[name] |= footprint(df, raw_size(name), CLEAN / "samples" / f"{name}.parquet")
    PROFILES[name]["yearly_raw_gb"] = yearly_volume_gb(PROFILES[name]["rows_per_day"], PROFILES[name]["raw_bytes_per_row"])

## 3. Bảng so sánh và dữ liệu sạch

- `compare(profiles: dict[str, dict]) -> DataFrame`: một dòng mỗi nguồn (index là tên nguồn), sắp theo
  `rows_per_day` giảm dần.
- `write_clean(table, out_dir)`: ghi đè `out_dir/profiles.parquet`, trả đường dẫn.

In [ ]:
def compare(profiles: dict[str, dict]) -> pd.DataFrame:
    """Một dòng mỗi nguồn, sắp theo rows_per_day giảm dần."""
    # TODO
    raise NotImplementedError


def write_clean(table: pd.DataFrame, out_dir: Path) -> Path:
    """Ghi đè out_dir/profiles.parquet, trả đường dẫn."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: bảng so sánh, ghi data/clean/source-profiles/profiles.parquet
TABLE = compare(PROFILES)
print(write_clean(TABLE, CLEAN))
TABLE[["rows", "rows_per_day", "median_interval_s", "max_gap_s", "duplicate_rows", "duplicate_keys",
       "worst_null_column", "worst_null_share", "raw_bytes_per_row", "compression_ratio", "yearly_raw_gb"]]

## 4. Biểu đồ

Ít nhất hai biểu đồ: velocity (`rows_per_day`, trục log) và dung lượng mỗi dòng thô vs Parquet theo nguồn.
Thêm: tỉ lệ thiếu tệ nhất mỗi nguồn.

In [ ]:
# explore: velocity
# TODO: barh rows_per_day theo nguồn, trục log

In [ ]:
# explore: dung lượng
# TODO: barh raw_bytes_per_row và parquet_bytes_per_row theo nguồn

## 5. Báo cáo

`report.md` cạnh notebook, 4 mục bắt buộc. Ở **Insight**, với mỗi nguồn đưa ra con số volume, velocity,
chất lượng, và **một khuyến nghị thu thập** (batch hay streaming, pull hay push, full hay incremental và
theo khoá nào, cảnh báo gì), có lý do dựa trên số đo.